<a href="https://colab.research.google.com/github/hamzafarooq/multi-agent-course/blob/main/modules/Module_3_Production_Agentic_RAG_AI_Systems/001.%20Agentic%20Router.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Dive Agentic Retrieval Augmented Generation

An Agentic RAG is required when we use reasoning to determine which action(s) to take and in which order to take them. Essentially we use agents instead of a LLM directly to accomplish a set of tasks which requires planning, multi step reasoning, tool use and/or learning over time. Agents give us agency!

Agency : The ability to take action or to choose what action to take

In the context of RAG, we can plug in agents to enhance the reasoning prior to selection of RAG pipelines, within a RAG pipeline for retrieval or reranking and finally for synthesising before we send out the response. This improves RAG to a large extent by automating complex workflows and decisions that are required for a non trivial RAG use case.

### Purpose of this Agentic RAG
This notebook presents a practical implementation of Agentic Retrieval-Augmented Generation (RAG)—a system where decision-making and tool selection are delegated to an intelligent agent before executing a response. Rather than passing every query through a static RAG pipeline, this system introduces agency—the ability to choose the best course of action depending on the nature of the query.

At the heart of this implementation is a router prompt, which classifies user queries into one of three categories:

- OpenAI documentation: Queries related to tools, APIs, or usage guidelines for OpenAI models
- 10-K financial reports: Questions requiring retrieval from company filings or financial datasets
- Live Internet search: Broader, current, or comparative queries that need web access

Once the query is classified, the system invokes a corresponding route handler:

- For OpenAI and 10-K queries, it retrieves relevant context from a vector database (Qdrant) using text embeddings, then applies a RAG-based response generator.
- For Internet queries, it fetches real-time information using a web-access API (ARES).

This approach is an example of Agentic RAG, where reasoning precedes retrieval and generation. By plugging in agents before and within the RAG pipeline, we make the system smarter and more adaptive. This allows us to:

- Automatically choose the right retrieval method based on context
- Combine structured knowledge with real-time search
- Scale RAG beyond trivial use cases by integrating multi-step decision logic

Importantly, no external agentic frameworks are used—this is a ground-up implementation that demonstrates how to build a lightweight but intelligent agentic system using only a language model, prompt engineering, and retrieval tools.

> **Reviewer guide.** Sections 1–6 are the course walkthrough. The assignment is at the end:
> **Part 1** (sub-query division), the **Bonus** (RBAC-aware semantic cache), then an
> **Evaluation** section (routing accuracy, compound queries, citations, concurrency benchmark,
> RBAC matrix) and **Final Results**. Reusable code lives in `src/`; offline tests in `tests/`
> run with `python -m pytest` (no API keys needed). See `README.md` for setup.


## Setup and Dependencies

In [1]:
# Install the necessary libraries
!pip install openai
!pip install qdrant_client
!pip install transformers==4.48.0

In [2]:
# Import basic libraries
import requests             # Used for making HTTP requests (e.g., calling ARES API for live internet queries)
import json                 # For parsing and structuring JSON data (especially OpenAI and routing responses)

# Credentials — Colab Secrets when on Colab, a local .env otherwise
try:
    from google.colab import userdata          # Colab: keys live in the 🔑 Secrets panel
    IN_COLAB = True
except ImportError:                            # Local Jupyter: keys live in .env
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv())
    IN_COLAB = False

    class userdata:                            # same .get() call works in both places
        @staticmethod
        def get(name):
            import os
            return os.getenv(name) or os.getenv(name.lower()) or os.getenv(
                name.replace("SERP_API_KEY", "SERPAPI_KEY"))

# OS operations
import os                   # Useful for accessing environment variables and managing paths

# OpenAI API client
from openai import OpenAI   # Official OpenAI client library to interface with GPT models for routing and generation

# Text processing
import re                   # Regular expressions for cleaning or preprocessing inputs (if needed)

# Optional visualization (for analysis/debugging purposes)
import matplotlib.pyplot as plt       # For displaying charts or visual debug outputs (e.g., embeddings visualizations)
import matplotlib.image as mpimg      # For loading/displaying images if needed (rare in RAG, but helpful in demos)

# Embedding models (used for text vectorization during retrieval)
from transformers import AutoTokenizer, AutoModel  # For loading custom transformer models if not using OpenAI embeddings

from qdrant_client import models

import qdrant_client
import asyncio
import nest_asyncio # Import nest_asyncio
nest_asyncio.apply() # Apply nest_asyncio to allow nested event loops
# # Vector database client
# from qdrant_client import QdrantClient   # Qdrant is used as the vector store to retrieve documents based on similarity

/private/tmp/claude-501/-Users-anuraglahon-Downloads/291f8e73-0c82-4691-a0f3-5381512d81fe/scratchpad/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# The assignment logic lives in the src/ package of this repo (tested offline with pytest);
# this notebook is the executable walkthrough. Make `import src` work in Colab and locally.
import sys

REPO_URL = "https://github.com/anuraglahon16/agentic-router-assignment.git"
if IN_COLAB:
    if not os.path.isdir("/content/agentic-router-assignment"):
        !git clone -q {REPO_URL} /content/agentic-router-assignment
    REPO_ROOT = "/content/agentic-router-assignment"
else:
    REPO_ROOT = os.getcwd()          # run the notebook from the repo root
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import src
print("src package:", os.path.dirname(src.__file__))


src package: /Users/anuraglahon/Downloads/agentic-router-assignment/src


## 1. Defining the Internet Tool

First, we will define a tool function that enables our system to answer queries requiring real-time, internet-based information. Not all questions can be answered using static documents like OpenAI docs or financial filings—sometimes users ask about current trends, comparisons, or live updates.

To handle this, we introduce a live search capability using the **SerpApi**.

### What is SerpApi?  
SerpApi is a Google Search API that allows you to:

- Search the internet in real time using Google.
- Get structured results including answer boxes, organic results, and snippets.

This is particularly useful for questions about:

- Current events (e.g., *"Latest AI tools in 2025"*),
- Tech comparisons (e.g., *"Gemini vs GPT-4"*),
- General knowledge outside internal datasets.

Please generate the API key [here](https://serpapi.com)


In [4]:
#loads serp api key from colab secrets
serp_api_key=userdata.get('SERP_API_KEY')

In [5]:
def get_internet_content(user_query: str, action: str):
    """
    Fetches a response from the internet using SerpApi based on the user's query.

    This function serves as the tool invoked when the router classifies a query
    as requiring real-time information beyond internal datasets—i.e., "INTERNET_QUERY".
    It sends the query to SerpApi (Google Search) and returns structured results.

    Args:
        user_query (str): The user's question that needs a live answer.
        action (str): Route type (always expected to be "INTERNET_QUERY").

    Returns:
        str: Response text from live Google search results or an error message.
    """
    print("Getting your response from the internet 🌐 ...")

    params = {
        "q": user_query,
        "api_key": serp_api_key,
        "engine": "google",
        "num": 5,
    }

    try:
        response = requests.get("https://serpapi.com/search.json", params=params)
        response.raise_for_status()
        data = response.json()

        parts = []

        # Answer box — Google's highlighted direct answer (most relevant)
        answer_box = data.get("answer_box", {})
        if answer_box.get("answer"):
            parts.append(f"[Direct Answer] {answer_box['answer']}")
        elif answer_box.get("snippet"):
            parts.append(f"[Direct Answer] {answer_box['snippet']}")

        # Top organic results — titles + snippets
        for i, result in enumerate(data.get("organic_results", [])[:5], start=1):
            title = result.get("title", "")
            snippet = result.get("snippet", "")
            link = result.get("link", "")
            if snippet:
                parts.append(f"[{i}] {title}\n    {snippet}\n    Source: {link}")

        if not parts:
            return "No results found."

        return "\n\n".join(parts)

    # Handle HTTP-level errors (e.g., 400s or 500s)
    except requests.exceptions.HTTPError as http_err:
        return f"HTTP error occurred: {http_err}"

    # Handle general connection, timeout, or request formatting issues
    except requests.exceptions.RequestException as req_err:
        return f"Request error occurred: {req_err}"

    # Catch-all for any unexpected failure
    except Exception as err:
        return f"An unexpected error occurred: {err}"


In [6]:
print(get_internet_content("Tell me about best travel destinations in 2026?","INTERNET_QUERY")) #run internet function to test results

Getting your response from the internet 🌐 ...


[1] Manila cracks prestigious list of best global travel ...
    1. The Dolomites, Milan, Italy · 2. Vancouver, Canada · 3. Beijing, China · 4. Rabat, Morocco · 5. Hull, Yorkshire, England · 6. Manila, Philippines.
    Source: https://www.postguam.com/lifestyle/world/manila-cracks-prestigious-list-of-best-global-travel-destinations/article_48f285fe-bd7c-4c84-bedf-f4381d353019.html

[2] Best US Beach Vacation Rentals for Summer 2026
    The top US beach vacation rental destinations for summer 2026 are Outer Banks NC, Gulf Shores AL, 30A/Destin FL, Cape Cod MA, Hilton Head SC, ...
    Source: https://www.houfy.com/blog/best-us-beach-vacation-rentals-for-summer-2026

[3] Travel + Leisure Names Brazil 2026 Destination of the Year
    Brazil 2026 Destination of the Year. These cities are best explored by foot, bike, or public transport.
    Source: https://www.tripadvisor.com/ShowTopic-g294280-i1045-k15482187-Travel_Leisure_Names_Brazil_2026_Destination_of_the_Year-Brazil.html

[4] Why Bhut

## 2. Router Query Function — Giving the Agent Its Brain

In this step, we will define the router function, which plays a critical role in our Agentic RAG system.

### What is a Router?

A router is like the decision-making brain of our assistant.

Before trying to answer a user's question, the system first needs to figure out:

> “Where should I go to find the right answer?”

To make this decision, we use the OpenAI GPT model. We provide it with a detailed system prompt that explains how to classify the user's question into one of these categories:

- **OPENAI_QUERY** → Questions about OpenAI tools, APIs, models, or documentation.
- **10K_DOCUMENT_QUERY** → Questions about companies, financial filings, or analysis based on 10-K reports.
- **INTERNET_QUERY** → Anything else that likely requires real-time or general web information.

### What does the function do?

- Sends the user's question to the OpenAI API.
- Receives a JSON response containing:
  - `action`: The category the query belongs to.
  - `reason`: A short explanation for the decision.
  - `answer`: (Optional) A quick response if it’s simple enough (left blank for internet queries).
- Parses the response and returns it as a Python dictionary.

### Why is this important?

This router gives the system agency—the ability to decide which knowledge source to use. It’s what makes this pipeline agentic, not just static.

Without the router, every query would follow the same path. With it, we can:

- Dynamically switch between tools and data sources.
- Handle different types of user questions intelligently.
- Avoid wasting resources on unnecessary steps.

**Update:** the router now asks for **structured output** — the OpenAI SDK validates the reply
against a Pydantic `RouteDecision` (`action` must be one of the three labels). The original
defensive JSON parser is kept as the fallback, and an unusable reply still falls back to
`INTERNET_QUERY` exactly as before. Implementation: `src/router.py`.


## Query Routing Workflow

The diagram below shows the full decision flow — from receiving a user query to returning a final response.

```
                        ┌─────────────────────┐
                        │     User Query      │
                        └──────────┬──────────┘
                                   │
                                   ▼
                    ┌──────────────────────────────┐
                    │      Router LLM (GPT-4o)     │
                    │         route_query()         │
                    │                              │
                    │  Reads the query and decides │
                    │  which data source to use    │
                    └──────────────┬───────────────┘
                                   │
           ┌───────────────────────┼───────────────────────┐
           │                       │                       │
           ▼                       ▼                       ▼
┌─────────────────────┐ ┌─────────────────────┐ ┌─────────────────────┐
│    OPENAI_QUERY     │ │ 10K_DOCUMENT_QUERY  │ │   INTERNET_QUERY    │
│                     │ │                     │ │                     │
│ e.g. "What are      │ │ e.g. "What was      │ │ e.g. "Best LLMs     │
│  OpenAI Agents?"    │ │  Uber's revenue?"   │ │  in 2026?"          │
└──────────┬──────────┘ └──────────┬──────────┘ └──────────┬──────────┘
           │                       │                        │
           ▼                       ▼                        ▼
┌─────────────────────┐ ┌─────────────────────┐  ┌──────────────────────┐
│   Embed Query       │ │   Embed Query        │  │      SerpApi         │
│  (Nomic Model)      │ │  (Nomic Model)       │  │  get_internet_       │
│                     │ │                      │  │  content()           │
│  get_text_          │ │  get_text_           │  │                      │
│  embeddings()       │ │  embeddings()        │  │  Live Google search  │
└──────────┬──────────┘ └──────────┬──────────┘  └──────────┬───────────┘
           │                       │                          │
           ▼                       ▼                          │
┌─────────────────────┐ ┌─────────────────────┐              │
│  Qdrant Vector DB   │ │  Qdrant Vector DB   │              │
│  Collection:        │ │  Collection:         │              │
│  "opnai_data"       │ │  "10k_data"          │              │
│                     │ │                      │              │
│  Retrieve top-3     │ │  Retrieve top-3      │              │
│  similar chunks     │ │  similar chunks      │              │
└──────────┬──────────┘ └──────────┬──────────┘              │
           │                       │                          │
           └───────────┬───────────┘                          │
                       ▼                                      │
           ┌───────────────────────┐                          │
           │    RAG Response       │                          │
           │    Generator          │                          │
           │  rag_formatted_       │                          │
           │  response()           │                          │
           │                       │                          │
           │  GPT-4 synthesizes    │                          │
           │  answer from context  │                          │
           │  + adds citations     │                          │
           └───────────┬───────────┘                          │
                       │                                      │
                       └──────────────────┬───────────────────┘
                                          ▼
                             ┌────────────────────────┐
                             │     Final Response     │
                             │       to User          │
                             └────────────────────────┘
```

### Key decision points at a glance

| Route | Trigger | Retrieval Method | Response Generator |
|---|---|---|---|
| `OPENAI_QUERY` | OpenAI docs, APIs, Agents | Qdrant `opnai_data` (top-3 chunks) | `rag_formatted_response()` via GPT-4 |
| `10K_DOCUMENT_QUERY` | Financial filings, company revenue | Qdrant `10k_data` (top-3 chunks) | `rag_formatted_response()` via GPT-4 |
| `INTERNET_QUERY` | Anything else / real-time info | SerpApi live Google search | Raw search result snippets |

> **Note:** Both vector-based routes share the same embedding model (`nomic-embed-text-v1.5`) and RAG generator — only the Qdrant collection changes. The router's JSON output (`action` field) is the single decision variable that drives the entire flow.


In [7]:
# Securely retrieve the OpenAI API key from Colab's user data store
# This avoids hardcoding sensitive credentials directly in the notebook
openai_api_key = userdata.get('OPENAI_API_KEY')

# Initialize the OpenAI client with the retrieved API key
# This client will be used for:
# - Query classification via the router prompt
# - Potentially generating responses from retrieved context
openaiclient = OpenAI(api_key=openai_api_key)


In [8]:
# The router prompt (unchanged) and its parsing live in src/router.py:
#   1. structured output — chat.completions.parse(response_format=RouteDecision) validates the JSON
#   2. defensive parser  — strips code fences / prose, extracts the JSON, validates the route label
#   3. fallback          — INTERNET_QUERY, exactly as the original router did on errors
from src.router import ROUTER_PROMPT, RouteDecision, route_query as _route_query


def route_query(user_query: str) -> dict:
    """
    Classify a query into OPENAI_QUERY / 10K_DOCUMENT_QUERY / INTERNET_QUERY.

    Returns the same dict as before ({"action", "reason", "answer"}) plus
    "parsed_by": "structured" | "parser" | "fallback".
    """
    return _route_query(user_query, openaiclient)


In [9]:
route_query("what is the revenue of uber in 2021?")


{'action': '10K_DOCUMENT_QUERY',
 'reason': "Revenue data from Uber's 2021 annual report",
 'answer': "See Uber's 2021 10-K",
 'parsed_by': 'structured'}

In [10]:
route_query("what is an AI Agent?")

{'action': 'OPENAI_QUERY',
 'reason': "AI agents are covered by OpenAI's official agent documentation",
 'answer': 'Software that performs tasks',
 'parsed_by': 'structured'}

## 3. Setting Up Qdrant Vector Database for Agentic RAG
In this step, we are connecting our agent to a pre-built vector database using Qdrant—a tool used to store and search document embeddings (numerical representations of text).

What Are We Doing?
We are loading an existing Qdrant database that was downloaded from a GitHub repository. This database already contains:

- Vectorized OpenAI documentation
- Vectorized 10-K financial filings

By loading this saved data:

- We save time (no need to re-embed the documents)
- We enable fast similarity search to retrieve relevant text chunks

This setup allows our system to perform semantic search, meaning it can understand the meaning of the user query and match it with the most relevant pieces of information stored in the database.


### Why This Matters in Agentic RAG
Once the router decides that the query should go to the OpenAI docs or the 10-K reports, our system uses Qdrant to:

- Search for the most relevant pieces of text
- Pass those to the model to generate a grounded answer

So, this step is essential to support retrieval-augmented generation (RAG) within our agentic flow.

#Data Sources:

**10K Database: Lyft 2024 & Uber 2021 SEC filings**

**OpenAI Docs: Official OpenAI documentation about Agents**

For lecture demo purposes, the vecitr database has already been created and hosted on Github which we will clone here. In order to create your own embeddings, the notebook and data will be hosted and shared on github

In [11]:
# Colab only — wipe a previous clone if you need a clean copy
if IN_COLAB:
    !rm -rf /content/multi-agent-course

In [12]:
# The prebuilt Qdrant collections (10-K + OpenAI docs) ship with the repo.
# On Colab we clone to get them; locally you already have them.
if IN_COLAB:
    !git clone https://github.com/hamzafarooq/multi-agent-course.git

In [13]:
# 🗄️ Initializing Qdrant client with the local path to the vector database
# Prebuilt collections (10-K and OpenAI docs) — cloned on Colab, already present locally.
_MODULE = "modules/Module_3_Production_Agentic_RAG_AI_Systems"
if os.getenv("QDRANT_PATH"):                 # explicit override, e.g. data checked out elsewhere
    QDRANT_PATH = os.getenv("QDRANT_PATH")
elif IN_COLAB:
    QDRANT_PATH = f"/content/multi-agent-course/{_MODULE}/Agentic_RAG/qdrant_data"
else:
    # the notebook lives in the module folder, so the data sits right next to it
    QDRANT_PATH = os.path.join(os.getcwd(), "Agentic_RAG", "qdrant_data")

print("Qdrant path:", QDRANT_PATH)
client = qdrant_client.AsyncQdrantClient(path=QDRANT_PATH)

Qdrant path: /private/tmp/claude-501/-Users-anuraglahon-Downloads/291f8e73-0c82-4691-a0f3-5381512d81fe/scratchpad/multi-agent-course/modules/Module_3_Production_Agentic_RAG_AI_Systems/Agentic_RAG/qdrant_data


## 4. Building the Retriever and RAG for Vector Databases
In this section, we build the core logic that allows our agent to find relevant documents and generate grounded answers using them.

###Step 1: Import the Embedding Model
We start by importing the nomic-ai/nomic-embed-text-v1.5 model from Hugging Face. This model is used to convert any text (such as a user query) into a dense vector, known as an embedding. These embeddings capture the semantic meaning of text, allowing us to later compare and retrieve similar documents.


In [14]:
# Load the tokenizer and embedding model from Hugging Face
# This model converts raw text into dense vector representations (embeddings)
# Used for similarity search in Qdrant during document retrieval
text_tokenizer = AutoTokenizer.from_pretrained("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)
text_model = AutoModel.from_pretrained("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)

def get_text_embeddings(text):
    """
    Converts input text into a dense embedding using the Nomic embedding model.
    These embeddings are used to query Qdrant for semantically relevant document chunks.

    Args:
        text (str): The input text or query from the user.

    Returns:
        np.ndarray: A fixed-size vector representing the semantic meaning of the input.
    """
    # Tokenize and prepare input for the model
    inputs = text_tokenizer(text, return_tensors="pt", padding=True, truncation=True)

    # Forward pass to get model outputs
    outputs = text_model(**inputs)

    # Take the mean across all token embeddings to get a single vector (pooled representation)
    embeddings = outputs.last_hidden_state.mean(dim=1)

    # Convert to NumPy array and detach from computation graph
    return embeddings[0].detach().numpy()

# Example usage: Generate and preview the embedding of a test sentence
text = "This is a test sentence."
embeddings = get_text_embeddings(text)
print(embeddings[:5])  # Print first 5 dimensions for inspection


<All keys matched successfully>


[ 1.2799684   0.4015834  -3.5162644  -0.39813134  1.5919116 ]


### Step 2: Define the Embedding Function
We then define a function get_text_embeddings() which:

- Tokenizes the input text
- Runs it through the model
- Computes the average of all token embeddings
- Returns a single vector that represents the full sentence

This vector will be used to query Qdrant to find the most relevant document chunks based on similarity.

In [15]:
def rag_formatted_response(user_query: str, context: list):
    """
    Generate a response to the user query using the provided context,
    with article references formatted as [1][2], etc.

    This function performs the final step in the RAG pipeline—synthesizing an answer
    from retrieved document chunks (context). It prompts the model to generate a
    grounded response, explicitly citing sources using a reference format.

    Args:
        user_query (str): The user's original question.
        context (list): List of text chunks retrieved from Qdrant (10-K or OpenAI docs).

    Returns:
        str: A generated response grounded in the retrieved context, with numbered citations.
    """

    # Construct a RAG prompt that includes both:
    # 1. The user's query
    # 2. The supporting context documents
    # The prompt instructs the model to answer using only the provided context,
    # and to include citations like [1], [2], etc. based on chunk IDs or order.
    rag_prompt = f"""
       Based on the given context, answer the user query: {user_query}\nContext:\n{context}
       and employ references to the ID of articles provided [ID], ensuring their relevance to the query.
       The referencing should always be in the format of [1][2]... etc. </instructions>
    """

    #  Call GPT-5.6-Luna to generate the response using the RAG-style prompt
    response = openaiclient.chat.completions.create(
        model="gpt-5.6-luna",
        messages=[
            {"role": "system", "content": rag_prompt},
        ]
    )

    # Return the model's generated answer
    return response.choices[0].message.content


### Step 3: Define the RAG Response Generator
After retrieving relevant text chunks from Qdrant, we use the rag_formatted_response() function to generate a final answer. This function:

- Takes the user query and the retrieved document chunks
- Builds a prompt that asks the language model (GPT-5.6-Luna) to answer the question using only the provided context
- Instructs the model to include references like [1], [2] for traceability

This ensures the output is not only informative but also grounded in actual retrieved data.

Together, these two functions lay the foundation for combining retrieval (from vector DB) and generation (from LLM) — the two pillars of a RAG system.



In [16]:
async def retrieve_and_response(user_query: str, action: str):
    """
    Retrieves relevant text chunks from the appropriate Qdrant collection
    based on the query type, then generates a response using RAG.

    This function powers the retrieval and response generation pipeline
    for queries that are classified as either OPENAI-related or 10-K related.
    It uses semantic search to fetch relevant context from a Qdrant vector store
    and then generates a response using that context via a RAG prompt.

    Args:
        user_query (str): The user's input question.
        action (str): The classification label from the router (e.g., "OPENAI_QUERY", "10K_DOCUMENT_QUERY").

    Returns:
        str: A model-generated response grounded in retrieved documents, or an error message.
    """

    # Define mapping of routing labels to their respective Qdrant collections
    collections = {
        "OPENAI_QUERY": "opnai_data",           # Collection of OpenAI documentation embeddings
        "10K_DOCUMENT_QUERY": "10k_data"        # Collection of 10-K financial document embeddings
    }

    try:
        # Ensure that the provided action is valid
        if action not in collections:
            return "Invalid action type for retrieval."

        # Step 1: Convert the user query into a dense vector (embedding)
        try:
            query = get_text_embeddings(user_query)
        except Exception as embed_err:
            return f"Embedding error: {embed_err}"  # Fail early if embedding fails

        # Step 2: Retrieve top-matching chunks from the relevant Qdrant collection
        try:
            text_hits = await client.query_points(
                collection_name=collections[action],  # Choose the right collection based on routing
                query=query,                          # The embedding of the user's query
                limit=3                               # Fetch top 3 relevant chunks
            )
        except Exception as qdrant_err:
            return f"Vector DB query error: {qdrant_err}"  # Handle Qdrant access issues

        # Extract the raw content from the retrieved vector hits
        contents = [point.payload['content'] for point in text_hits.points]

        # If no relevant content is found, return early
        if not contents:
            return "No relevant content found in the database."

        # Step 3: Pass the retrieved context to the RAG model to generate a response
        try:
            response = rag_formatted_response(user_query, contents)
            return response
        except Exception as rag_err:
            return f"RAG response error: {rag_err}"  # Handle generation failures

    # Catch any unforeseen errors in the overall process
    except Exception as err:
        return f"Unexpected error: {err}"


# 5. Putting It All Together: Running the Agentic RAG
In this final step, we combine everything into a single function that controls the entire Agentic RAG workflow. The agentic_rag() function acts as the main orchestrator of the system.

Here’s what it does:

- Prints the user's query for reference.
- Uses the router function (powered by GPT) to decide which type of data source to use:
  - OpenAI documentation
  - 10-K financial reports
- Internet search
- Calls the correct function based on the route:
- If it’s an OpenAI or 10-K query, it retrieves data from Qdrant and generates a RAG response.
- If it’s an Internet query, it uses the ARES API to fetch live information.
- Displays the final response, neatly formatted in the console.

This step brings the agentic loop full circle—from understanding the question, reasoning about where to search, to finally responding with the best possible answer.

In [17]:
# Dictionary that maps the route labels (decided by the router) to their respective functions
# Each type of query is handled differently:
# - OPENAI_QUERY and 10K_DOCUMENT_QUERY use document retrieval + RAG
# - INTERNET_QUERY uses a web search API
routes = {
    "OPENAI_QUERY": retrieve_and_response,
    "10K_DOCUMENT_QUERY": retrieve_and_response,
    "INTERNET_QUERY": get_internet_content,
}

def agentic_rag(user_query: str):
    """
    Main function that runs the full Agentic RAG system.

    This function takes a user's question, decides what type of query it is (OpenAI-related,
    financial document-related, or general internet), and then calls the right function
    to handle it. Finally, it prints out the full conversation and response.

    Args:
        user_query (str): The user's input question.

    Returns:
        None (It just prints the result nicely to the console)
    """

    #  Terminal color codes to make the printed output easier to read and visually structured
    CYAN = "\033[96m"
    GREY = "\033[90m"
    BOLD = "\033[1m"
    RESET = "\033[0m"

    try:
        # Step 1: Print the user's original question to the console
        print(f"{BOLD}{CYAN}👤 User Query:{RESET} {user_query}\n")

        # Step 2: Use the router (powered by GPT) to decide which route the query belongs to
        try:
            response = route_query(user_query)
        except Exception as route_err:
            # If something goes wrong while classifying the query, show an error message
            print(f"{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\n")
            print(f"Routing error: {route_err}\n")
            return

        # Extract the routing decision and the reason behind it
        action = response.get("action")  # e.g., "OPENAI_QUERY"
        reason = response.get("reason")  # e.g., "Related to OpenAI tools"

        # Step 3: Show the selected route and why it was chosen
        print(f"{GREY}📍 Selected Route: {action}")
        print(f"📝 Reason: {reason}")
        print(f"⚙️ Processing query...{RESET}\n")

        # Step 4: Call the correct function depending on the route (retrieval or web search)
        try:
            route_function = routes.get(action)  # Find the function to use for this route
            if route_function:
                if action in ["OPENAI_QUERY", "10K_DOCUMENT_QUERY"]:
                    # Use asyncio.run for async functions, nest_asyncio will handle nested loops
                    result = asyncio.run(route_function(user_query, action))
                else:
                    # Otherwise, call it directly (e.g., get_internet_content is synchronous)
                    result = route_function(user_query, action)
            else:
                result = f"Unsupported action: {action}"  # Catch unknown routing types
        except Exception as exec_err:
            result = f"Execution error: {exec_err}"  # Handle failure in the chosen route function

        # Step 5: Print the final response to the user
        print(f"{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\n")
        print(f"{result}\n")

    except Exception as err:
        # Catch-all for any unexpected errors in the overall logic
        print(f"{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\n")
        print(f"Unexpected error occurred: {err}\n")


In [18]:
agentic_rag("what was uber revenue in 2021?")

👤 User Query: what was uber revenue in 2021?



📍 Selected Route: 10K_DOCUMENT_QUERY
📝 Reason: Uber's 2021 revenue is reported in its annual financial documents.
⚙️ Processing query...



🤖 BOT RESPONSE:

Uber’s revenue in 2021 was **$17.455 billion** (approximately **$17.5 billion**). [2]



In [19]:
agentic_rag("what was lyft revenue in 2022?")

👤 User Query: what was lyft revenue in 2022?



📍 Selected Route: 10K_DOCUMENT_QUERY
📝 Reason: Asks for Lyft's annual revenue, a financial fact typically found in 10-K filings.
⚙️ Processing query...



🤖 BOT RESPONSE:

Lyft’s revenue in 2022 was **$4.095 billion** (approximately **$4,095,135,000**), up 28% from 2021.[1]



In [20]:
agentic_rag("List me down new LLMs in 2025")

👤 User Query: List me down new LLMs in 2025



📍 Selected Route: INTERNET_QUERY
📝 Reason: Requests a current list of newly released LLMs, requiring broader web research.
⚙️ Processing query...

Getting your response from the internet 🌐 ...
🤖 BOT RESPONSE:

[1] llm-wiki · GitHub
    A pattern for building personal knowledge bases using LLMs. This is an idea file, it is designed to be copy pasted to your own LLM Agent (e.g. OpenAI Codex, ...
    Source: https://gist.github.com/karpathy/442a6bf555914893e9891c11519de94f?permalink_comment_id=6325337

[2] Private AI vs Claude: 12 Models Compared | Everyday AI
    I ran 12 models through an AI research pipeline to compare private and public AI. Private models scored within 2 points of Claude.
    Source: https://medium.com/everyday-ai/i-built-a-barebone-ai-research-harness-to-compare-private-and-public-models-heres-what-happened-8da25b456959

[3] 2026 Tech Goals: RDMA, vLLM, Monarch, and more
    Manually defining llm traces with OTel / tempo. Looks like I could get a lot of this for free

In [21]:
agentic_rag("how to work with chat completions?")

👤 User Query: how to work with chat completions?



📍 Selected Route: OPENAI_QUERY
📝 Reason: Chat Completions is an OpenAI API
⚙️ Processing query...



🤖 BOT RESPONSE:

Chat Completions let you send a sequence of messages to a model and receive an assistant response. You manage the conversation history yourself by adding each new user and assistant message to the `messages` array.

### Basic example

```python
from openai import OpenAI

client = OpenAI()

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        {
            "role": "user",
            "content": "Explain how APIs work."
        }
    ]
)

answer = response.choices[0].message.content
print(answer)
```

### Continuing a conversation

Include previous messages in every request:

```python
messages = [
    {"role": "system", "content": "You are a concise tutor."},
    {"role": "user", "content": "What is recursion?"},
    {"role": "assistant", "content": "Recursion is when a function calls itself."},
    {"role": "user", "content": "C

In [22]:
agentic_rag("best ways to build Agents")

👤 User Query: best ways to build Agents



📍 Selected Route: OPENAI_QUERY
📝 Reason: Building agents is covered by OpenAI agent documentation
⚙️ Processing query...



🤖 BOT RESPONSE:

## Best ways to build agents

1. **Start with a suitable use case**  
   Build an agent when the task involves a multi-step workflow that benefits from independent decision-making and tool use—such as resolving support issues, booking reservations, generating reports, or modifying code. A simple chatbot or single-turn classifier does not necessarily need an agent. [1]

2. **Use the three core components**
   - **Model:** Handles reasoning and workflow decisions.
   - **Tools:** APIs or functions the agent can call to retrieve information or take actions.
   - **Instructions:** Define the agent’s behavior, constraints, and workflow. [1]

3. **Prototype with the most capable model first**  
   Establish an accuracy and reliability baseline using a strong model. Then evaluate whether smaller, faster, and less expensive models can handle individual tasks without unacceptable quality loss. [1]

4. **Define tools narrowly and clearly**  
   Each tool should have a well-defin

## 6. Role-Based Access Control (RBAC)

Everything so far assumes one kind of user: whoever asks gets whatever the router
picks. In a real deployment that's rarely true. An engineer shouldn't be able to pull
finance's 10-K numbers out of the vector store, and a finance analyst has no business
reading internal engineering docs — even though both are talking to the same agent.

RBAC puts a **permission check between the router's decision and the tool call**. The
router still reasons about *where* the answer lives; RBAC decides whether *this user*
is allowed to go there. If not, the request is rejected before any embedding, vector
search, or grounding call happens.

**This demo — 2 roles, 3 knowledge sources:**

| Knowledge source | Route label | `engineer` | `finance_analyst` |
|---|---|---|---|
| 📘 OpenAI documentation (Qdrant) | `OPENAI_QUERY` | ✅ | ✅ |
| 📗 10-K filings (Qdrant) | `10K_DOCUMENT_QUERY` | ❌ | ✅ |
| 🌐 Live internet search (SerpApi) | `INTERNET_QUERY` | ✅ | ❌ |

The two Qdrant collections and the SerpApi tool are the same ones built above — RBAC
is a layer on top, not a different pipeline.


In [23]:
# Users, roles and permissions now live in src/auth.py (same values as before) so the
# cache and the tests share one source of truth:
#
#   USERS            = {"alice": "engineer", "bob": "finance_analyst"}
#   ROLE_PERMISSIONS = {"engineer":        {"OPENAI_QUERY", "INTERNET_QUERY"},
#                       "finance_analyst": {"OPENAI_QUERY", "10K_DOCUMENT_QUERY"}}
#
# ROLE_PERMISSIONS is an allow-list: anything not listed is denied by default.
# In production USERS comes from SSO/JWT claims, never a dict in the notebook.
from src.auth import USERS, ROLE_PERMISSIONS, SOURCE_LABELS, has_access, allowed_sources

print("alice  (engineer)        →", allowed_sources("alice"))
print("bob    (finance_analyst) →", allowed_sources("bob"))
print("carol  (unknown user)    →", allowed_sources("carol"))


alice  (engineer)        → {'OPENAI_QUERY', 'INTERNET_QUERY'}
bob    (finance_analyst) → {'10K_DOCUMENT_QUERY', 'OPENAI_QUERY'}
carol  (unknown user)    → set()


In [24]:
def secure_agentic_rag(user_id: str, user_query: str):
    """
    The same agentic RAG loop as above, with one addition: after the router picks a
    route, the user's role must permit that route before the tool is called.

    Order of operations:
        1. Identify the user  → unknown users are rejected immediately
        2. Route the query    → router decides which knowledge source fits
        3. RBAC check         → role allowed to use that source? deny if not
        4. Retrieve + answer  → only ever reached by an authorized request

    Args:
        user_id (str): Who is asking (looked up in USERS).
        user_query (str): The question.

    Returns:
        str: The answer, or a denial message.
    """
    CYAN, GREY, RED, GREEN, BOLD, RESET = (
        "\033[96m", "\033[90m", "\033[91m", "\033[92m", "\033[1m", "\033[0m"
    )

    role = USERS.get(user_id)
    print(f"{BOLD}{CYAN}👤 User:{RESET} {user_id}  (role: {role or 'UNKNOWN'})")
    print(f"{BOLD}{CYAN}❓ Query:{RESET} {user_query}\n")

    # Step 1 — unknown identity is denied before anything else runs
    if role is None:
        print(f"{RED}🚫 ACCESS DENIED{RESET} — unknown user '{user_id}'.\n")
        return f"🚫 Access denied: unknown user '{user_id}'."

    # Step 2 — the router still does the reasoning about where the answer lives
    try:
        decision = route_query(user_query)
    except Exception as route_err:
        return f"Routing error: {route_err}"

    action = decision.get("action")
    reason = decision.get("reason")
    print(f"{GREY}📍 Selected Route: {action}")
    print(f"📝 Reason: {reason}{RESET}\n")

    # Step 3 — the gate. Nothing is embedded, searched, or grounded past this point
    #          unless the role is permitted to use the chosen source.
    if not has_access(user_id, action):
        source = SOURCE_LABELS.get(action, action)
        print(f"{RED}🚫 ACCESS DENIED{RESET} — role '{role}' may not query {source}.\n")
        return (
            f"🚫 Access denied: your role ('{role}') does not have permission to "
            f"query {source}."
        )

    print(f"{GREEN}✅ Access granted{RESET} — processing...\n")

    # Step 4 — identical to agentic_rag() from Section 5
    try:
        route_function = routes.get(action)
        if not route_function:
            return f"Unsupported action: {action}"
        if action in ["OPENAI_QUERY", "10K_DOCUMENT_QUERY"]:
            result = asyncio.run(route_function(user_query, action))
        else:
            result = route_function(user_query, action)
    except Exception as exec_err:
        result = f"Execution error: {exec_err}"

    print(f"{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\n")
    print(f"{result}\n")
    return result

### Demo — same question, different roles

Each pair below sends the *identical* query as `alice` (engineer) and `bob`
(finance_analyst). The router makes the same decision both times — only the
permission check differs.


In [25]:
print("=" * 70)
print("1) alice (engineer) asks about the 10-K — FINANCE-ONLY → DENIED")
print("=" * 70)
secure_agentic_rag("alice", "what was uber revenue in 2021?")

1) alice (engineer) asks about the 10-K — FINANCE-ONLY → DENIED
👤 User: alice  (role: engineer)
❓ Query: what was uber revenue in 2021?



📍 Selected Route: 10K_DOCUMENT_QUERY
📝 Reason: Uber's 2021 revenue is reported in its annual financial documents.

🚫 ACCESS DENIED — role 'engineer' may not query 10-K financial filings.



"🚫 Access denied: your role ('engineer') does not have permission to query 10-K financial filings."

In [26]:
print("=" * 70)
print("2) bob (finance_analyst) asks the same question → ALLOWED")
print("=" * 70)
secure_agentic_rag("bob", "what was uber revenue in 2021?")

2) bob (finance_analyst) asks the same question → ALLOWED
👤 User: bob  (role: finance_analyst)
❓ Query: what was uber revenue in 2021?



📍 Selected Route: 10K_DOCUMENT_QUERY
📝 Reason: Asks about a company's annual financial revenue, typically found in 10-K reports.

✅ Access granted — processing...



🤖 BOT RESPONSE:

Uber’s revenue in 2021 was **$17.455 billion** (approximately **$17.5 billion**). [1]



'Uber’s revenue in 2021 was **$17.455 billion** (approximately **$17.5 billion**). [1]'

In [27]:
print("=" * 70)
print("3) bob (finance_analyst) asks for live web results — ENGINEER-ONLY → DENIED")
print("=" * 70)
secure_agentic_rag("bob", "List me down new LLMs in 2025")

3) bob (finance_analyst) asks for live web results — ENGINEER-ONLY → DENIED
👤 User: bob  (role: finance_analyst)
❓ Query: List me down new LLMs in 2025



📍 Selected Route: INTERNET_QUERY
📝 Reason: Requires current information about LLM releases beyond OpenAI-specific documentation.

🚫 ACCESS DENIED — role 'finance_analyst' may not query live internet search.



"🚫 Access denied: your role ('finance_analyst') does not have permission to query live internet search."

In [28]:
print("=" * 70)
print("4) alice (engineer) asks about OpenAI docs — SHARED → ALLOWED")
print("=" * 70)
secure_agentic_rag("alice", "best ways to build Agents")

4) alice (engineer) asks about OpenAI docs — SHARED → ALLOWED
👤 User: alice  (role: engineer)
❓ Query: best ways to build Agents



📍 Selected Route: OPENAI_QUERY
📝 Reason: Building agents relates to OpenAI Agents documentation and tools

✅ Access granted — processing...



🤖 BOT RESPONSE:

## Best ways to build reliable agents

1. **Use an agent only when it adds value**  
   Build an agent when a task involves multiple steps, decisions, tool use, or adapting to changing conditions. A simple chatbot, classifier, or fixed workflow may be better served by conventional software [1].

2. **Start with the three core components**
   - **Model:** Handles reasoning and workflow decisions.
   - **Tools:** APIs, functions, databases, search, or other systems the agent can use.
   - **Instructions:** Define behavior, decision rules, and safety constraints [1].

3. **Prototype with the most capable model first**  
   Establish an accuracy and reliability baseline using a strong model. Then test smaller, faster, less expensive models for simpler subtasks such as retrieval, classification, or routing [1].

4. **Design tools carefully**
   - Give each tool a clear, narrow purpose.
   - Use explicit names, descriptions, parameters, and return values.
   - Ensure tools p

'## Best ways to build reliable agents\n\n1. **Use an agent only when it adds value**  \n   Build an agent when a task involves multiple steps, decisions, tool use, or adapting to changing conditions. A simple chatbot, classifier, or fixed workflow may be better served by conventional software [1].\n\n2. **Start with the three core components**\n   - **Model:** Handles reasoning and workflow decisions.\n   - **Tools:** APIs, functions, databases, search, or other systems the agent can use.\n   - **Instructions:** Define behavior, decision rules, and safety constraints [1].\n\n3. **Prototype with the most capable model first**  \n   Establish an accuracy and reliability baseline using a strong model. Then test smaller, faster, less expensive models for simpler subtasks such as retrieval, classification, or routing [1].\n\n4. **Design tools carefully**\n   - Give each tool a clear, narrow purpose.\n   - Use explicit names, descriptions, parameters, and return values.\n   - Ensure tools p

**Where this is still weak — and how you'd harden it:**

- **The router runs before the check.** One LLM call is spent classifying a query the
  user may not be allowed to ask. That's cheap and leaks nothing, but you can do
  better: pass `allowed_sources(user_id)` into the router prompt so it only ever
  chooses from routes the role can reach, and deny anything that falls outside.
- **This gates whole sources, not chunks.** When one collection mixes content that
  different roles may only *partially* see, push the check into the vector store with
  **payload-based filters** (Qdrant supports this natively) so restricted chunks never
  enter the retrieved context in the first place. You'll do exactly this, at file
  granularity, in `003. Agentic Router_semantic_caching_rbac.ipynb`.
- **Caching and RBAC interact badly if you're careless.** A shared semantic cache
  keyed only on the question will happily serve `bob`'s finance answer to `alice`.
  Any cache sitting behind an access check must be partitioned by role (or by the
  permitted source set) — think about this before you add one.
- **Every check is an audit point.** Log who asked for what and whether it was
  allowed; that trail is what makes the system defensible in a regulated environment.


# Assignment

**Required:** Part 1 — sub-query division. This is the graded piece for this notebook.

**Bonus (optional, ungraded):** RBAC with a semantic cache. It extends Section 6 and is a
useful warm-up for **ARGUS**, where you build multi-source retrieval with a real caching
layer and have to report cost with and without the cache.

| | Task | Status | Builds on |
|---|---|---|---|
| **Part 1** | Sub-query division | **Required** | Sections 2 & 5 |
| **Bonus** | RBAC + semantic cache, without cross-role leakage | Optional | Section 6 |

**Deliverable:** this notebook, run end to end, with Part 1 implemented in the stub cell.
If you take the bonus, include it in the same notebook with the self-check passing.


---

## Part 1 — Sub-query division

Right now a compound question is treated as one search. Ask *"What was Uber's revenue
in 2021 and what was Lyft's in 2024?"* and the router picks a single route and fires a
single retrieval — so you get a partial answer, or a muddled one.

Your job: break compound queries into focused sub-queries, run each one through the
full agentic pipeline independently, then compose a single coherent answer.

**Requirements**

1. Write `agentic_rag_multi(user_query)` that:
   - calls `sub_queries()` to split the query (reference implementation below),
   - **routes each sub-query separately** — they may legitimately land on different
     sources (one on `10K_DOCUMENT_QUERY`, another on `INTERNET_QUERY`),
   - collects the per-sub-query answers and synthesises **one** final response,
   - preserves citations from each sub-answer in the composed output.
2. Handle the single-question case without regression — one question in, one route,
   no extra LLM calls beyond the split.
3. Parse the model's JSON defensively. `sub_queries()` returns a *string*; it can come
   back wrapped in prose or a code fence. Don't let a malformed split crash the agent —
   fall back to treating the input as one query.

**Check yourself against these**

| Query | Expected behaviour |
|---|---|
| `"what was uber revenue in 2021?"` | 1 sub-query, 1 route, same as `agentic_rag()` |
| `"what was lyft revenue in 2021 and what was uber revenue in 2021"` | 2 sub-queries, both `10K_DOCUMENT_QUERY` |
| `"what was uber's 2021 revenue and what are the newest LLMs?"` | 2 sub-queries, **different** routes |

**Stretch:** run the sub-queries concurrently with `asyncio.gather` instead of in
sequence, and compare wall-clock time.


In [29]:
#Reference Code for sub query division (For Guidance Only)

def sub_queries(user_query):
  sub_queries_prompt= f"""
  You are a query router. If the input contains multiple distinct questions, break it into sub-questions. Otherwise, keep it as one. Return a JSON object like:

  {{
      "subQuestions": ["..."]
  }}


  Query: "{user_query}"
  Output:
  """
  response = openaiclient.chat.completions.create(
        model="gpt-5.6-luna",
        messages=[
            {"role": "system", "content": sub_queries_prompt},
        ]
    )
  return response.choices[0].message.content


In [30]:
print(sub_queries("what was lyft revenue in 2021 and what was uber revenue in 2021"))

{"subQuestions":["What was Lyft's revenue in 2021?","What was Uber's revenue in 2021?"]}


### Part 1 — implementation

```
user query ─► split (structured → defensive parser → [query]) ─► for each sub-query:
                 route independently ─► retrieve / web search ─► answer with [n] citations
          ─► namespace [n] → [k.n] ─► compose (1 LLM call) ─► verify every sub-answer is still cited
                                           └─ citations dropped / LLM error ─► deterministic sections
```

- **Single question:** 1 split call + the normal route/answer calls — no composition call.
- **Defensive split:** structured output first; code fences, prose, bare lists and alternative
  keys are handled by the parser; anything unusable means "one question". Duplicates are removed
  and at most `MAX_SUB_QUERIES` (5) sub-queries run.
- **Concurrency (stretch):** `concurrent=True` runs each sub-query's route + retrieval in its
  own thread via `asyncio.gather(asyncio.to_thread(...))`.
- Code: `src/splitter.py`, `src/router.py`, `src/citations.py`, `src/composer.py`, `src/pipeline.py`.


In [31]:
# ── Part 1: implementation ───────────────────────────────────────────────────
# The logic lives in src/ (unit-tested offline with pytest); these wrappers bind it to this
# notebook's OpenAI client and the Section 5 route handlers, keeping the original names.
import time

from src.splitter import MAX_SUB_QUERIES, QuerySplit, parse_sub_queries, split_query as _split_query
from src.citations import namespace_citations
from src.composer import compose_answer as _compose_answer
from src.retrieval import execute_route as _execute_route
from src.pipeline import (AgentContext, agentic_rag_multi as _agentic_rag_multi,
                          answer_sub_query as _answer_sub_query)

# One context object carries the OpenAI client, the model and the route handlers.
ctx = AgentContext(llm=openaiclient, routes=routes, route_fn=route_query)


def split_query(user_query: str) -> list[str]:
    """Structured split → defensive parser → [user_query] (never raises)."""
    return _split_query(user_query, openaiclient)


def execute_route(user_query: str, action: str) -> str:
    """Step 4 of agentic_rag(): run the chosen handler (awaiting async ones)."""
    return _execute_route(user_query, action, routes)


def answer_sub_query(sub_query: str) -> dict:
    """Route ONE sub-query on its own and answer it from the chosen source."""
    return _answer_sub_query(ctx, sub_query)


def compose_answer(user_query: str, parts: list[dict]) -> str:
    """Merge sub-answers with namespaced citations; deterministic fallback if citations drop."""
    return _compose_answer(user_query, parts, openaiclient)


def agentic_rag_multi(user_query: str, concurrent: bool = False, return_details: bool = False):
    """Split → route each sub-query independently → answer → compose one response."""
    return _agentic_rag_multi(ctx, user_query, concurrent=concurrent, return_details=return_details)


### Part 1 — checks

The three cases from the table. Watch the `🔀` line (how many sub-queries) and the `📍` lines (one route per sub-query).

In [32]:
# Case 1 — one question: 1 sub-query, 1 route, answer returned as-is (no composition call)
agentic_rag_multi("what was uber revenue in 2021?")

👤 User Query: what was uber revenue in 2021?



🔀 1 sub-query (structured):
   1. What was Uber's revenue in 2021?



📍 [1] Route: 10K_DOCUMENT_QUERY  —  Uber's 2021 revenue is reported in its annual report.

🤖 BOT RESPONSE:

Uber’s revenue in 2021 was **$17.455 billion** (approximately **$17.5 billion**). [1]



'Uber’s revenue in 2021 was **$17.455 billion** (approximately **$17.5 billion**). [1]'

In [33]:
# Case 2 — two questions, both answered from the 10-K collection
agentic_rag_multi("what was lyft revenue in 2021 and what was uber revenue in 2021")

👤 User Query: what was lyft revenue in 2021 and what was uber revenue in 2021



🔀 2 sub-queries (structured):
   1. What was Lyft's revenue in 2021?
   2. What was Uber's revenue in 2021?



📍 [1] Route: 10K_DOCUMENT_QUERY  —  Asks about Lyft's annual financial revenue.
📍 [2] Route: 10K_DOCUMENT_QUERY  —  Uber's revenue is financial information from annual reports.



🤖 BOT RESPONSE:

In 2021:

- **Lyft’s revenue:** **$4.095 billion** (approximately $4,095,135,000).[1.1]
- **Uber’s revenue:** **$17.455 billion** (approximately $17.5 billion).[2.1]

Citation key:
[1.x] → sub-question 1 (10-K financial filings): What was Lyft's revenue in 2021?
[2.x] → sub-question 2 (10-K financial filings): What was Uber's revenue in 2021?



"In 2021:\n\n- **Lyft’s revenue:** **$4.095 billion** (approximately $4,095,135,000).[1.1]\n- **Uber’s revenue:** **$17.455 billion** (approximately $17.5 billion).[2.1]\n\nCitation key:\n[1.x] → sub-question 1 (10-K financial filings): What was Lyft's revenue in 2021?\n[2.x] → sub-question 2 (10-K financial filings): What was Uber's revenue in 2021?"

In [34]:
# Case 3 — two questions routed to DIFFERENT sources (10-K + internet)
agentic_rag_multi("what was uber's 2021 revenue and what are the newest LLMs?")

👤 User Query: what was uber's 2021 revenue and what are the newest LLMs?



🔀 2 sub-queries (structured):
   1. What was Uber's 2021 revenue?
   2. What are the newest LLMs?



Getting your response from the internet 🌐 ...
📍 [1] Route: 10K_DOCUMENT_QUERY  —  Uber's 2021 revenue is financial information from an annual report.
📍 [2] Route: INTERNET_QUERY  —  Requires current information about newly released language models broadly.



🤖 BOT RESPONSE:

Uber reported **$17.455 billion in revenue in 2021**—approximately **$17.5 billion**, representing a **57% increase from 2020**.[1.1][1.2]

The provided search results do **not identify a definitive set of the newest LLMs**. They mostly reference general discussions about new or open-source models, including developments involving Mistral, Microsoft, and GPT-3.5, but do not provide a reliable, current ranking or list of the latest models.[2.1][2.2][2.3][2.4][2.5]

Citation key:
[1.x] → sub-question 1 (10-K financial filings): What was Uber's 2021 revenue?
[2.x] → sub-question 2 (live internet search): What are the newest LLMs?



"Uber reported **$17.455 billion in revenue in 2021**—approximately **$17.5 billion**, representing a **57% increase from 2020**.[1.1][1.2]\n\nThe provided search results do **not identify a definitive set of the newest LLMs**. They mostly reference general discussions about new or open-source models, including developments involving Mistral, Microsoft, and GPT-3.5, but do not provide a reliable, current ranking or list of the latest models.[2.1][2.2][2.3][2.4][2.5]\n\nCitation key:\n[1.x] → sub-question 1 (10-K financial filings): What was Uber's 2021 revenue?\n[2.x] → sub-question 2 (live internet search): What are the newest LLMs?"

### Stretch — sequential vs. concurrent

`concurrent=True` runs each sub-query's route + retrieval in its own thread via `asyncio.gather`. The split and the final composition are sequential in both modes, so the saving is roughly the time of all but the slowest sub-query.

In [35]:
q = "what was uber's 2021 revenue and what are the newest LLMs?"

t0 = time.perf_counter()
agentic_rag_multi(q)
sequential_s = time.perf_counter() - t0

t0 = time.perf_counter()
agentic_rag_multi(q, concurrent=True)
concurrent_s = time.perf_counter() - t0

print(f"Sequential: {sequential_s:.1f}s | Concurrent: {concurrent_s:.1f}s | "
      f"speed-up ×{sequential_s / concurrent_s:.2f}")

👤 User Query: what was uber's 2021 revenue and what are the newest LLMs?



🔀 2 sub-queries (structured):
   1. What was Uber's revenue in 2021?
   2. What are the newest large language models (LLMs)?



Getting your response from the internet 🌐 ...
📍 [1] Route: 10K_DOCUMENT_QUERY  —  Query asks for a company's annual financial data.
📍 [2] Route: INTERNET_QUERY  —  Requires current information about models across providers



🤖 BOT RESPONSE:

Uber’s 2021 revenue was **$17.455 billion**, or approximately **$17.5 billion**. [1.1]

Regarding the newest large language models (LLMs), the provided search results point to a 2025 overview featuring **OpenAI’s GPT-5**, **Anthropic’s Claude**, and **AlpineGate’s AlbertAGPT**. [2.1] LLMs are AI models trained to understand and generate human language. [2.2] However, the supplied information does not establish a definitive, up-to-date ranking of the newest models or provide specific release details for each one.

Citation key:
[1.x] → sub-question 1 (10-K financial filings): What was Uber's revenue in 2021?
[2.x] → sub-question 2 (live internet search): What are the newest large language models (LLMs)?

👤 User Query: what was uber's 2021 revenue and what are the newest LLMs?



🔀 2 sub-queries (structured):
   1. What was Uber's revenue in 2021?
   2. What are the newest large language models (LLMs)?



Getting your response from the internet 🌐 ...


📍 [1] Route: 10K_DOCUMENT_QUERY  —  Uber's 2021 revenue is reported in its annual 10-K filing.
📍 [2] Route: INTERNET_QUERY  —  Requires current information across multiple AI providers



🤖 BOT RESPONSE:

Uber’s 2021 revenue was **$17.455 billion** (approximately **$17.5 billion**). [1.1]

For the newest large language models, the provided search result points to a 2025 list featuring **OpenAI’s GPT-5, Anthropic’s Claude models, and AlpineGate’s AlbertAGPT**, among others. [2.1] However, the supplied information does not provide a complete or independently verified ranking of the newest LLMs. LLMs are AI systems trained to process and generate human language. [2.2]

Citation key:
[1.x] → sub-question 1 (10-K financial filings): What was Uber's revenue in 2021?
[2.x] → sub-question 2 (live internet search): What are the newest large language models (LLMs)?

Sequential: 6.9s | Concurrent: 5.7s | speed-up ×1.20


---

## Bonus (optional) — RBAC with a semantic cache

*Not graded on its own — but this is rehearsal for graded work. **ARGUS**, the full-stack
assignment for this module, requires a real caching layer and a cost panel showing spend per
100 sources with and without it. Build the cache here, on a system you already understand,
and you'll be extending it in ARGUS rather than meeting it for the first time under a deadline.*

> **Come back to this after notebooks 002 and 003** — `002. Semantic Caching.ipynb`
> for how a FAISS cache works, and `003. Agentic Router_semantic_caching_rbac.ipynb` for `SemanticCaching`
> in `rag_helpers.py` and the file-level RBAC gate you'll be extending. You can read the spec now;
> you'll have every piece you need once those two are done.

Section 6 gates access by role. A semantic cache makes repeat questions near-instant.
Put them together naively and you build a data leak: the cache is keyed on the
*question*, so once `bob` (finance_analyst) asks about Uber's revenue, `alice`
(engineer) asks the same thing, hits the cache, and is handed finance data the RBAC
gate was supposed to deny her — without a single retrieval ever running.

Your job: add caching to `secure_agentic_rag()` so that repeat questions are fast
**and** no answer ever crosses a permission boundary.

**Requirements**

1. Build a role-aware cache. Two viable designs — pick one and justify it in a comment:
   - **Partitioned:** a separate FAISS index (or a namespace) per role, so a lookup
     can only ever see entries its own role produced.
   - **Tagged:** one index, but each entry stores the role (or the permitted source
     set) that produced it, and a hit is only honoured when it matches the caller.
2. Write `secure_agentic_rag_cached(user_id, user_query)` with this order of
   operations — it matters:
   ```
   unknown user      → DENIED   (no embedding, no cache, no LLM)
   route the query   → which source does this need?
   role not allowed  → DENIED   (still no cache lookup — a denial must not be cacheable)
   cache lookup      → HIT  → return stored answer
                     → MISS → run the pipeline, store, return
   ```
3. Return a dict, not a bare string, so the self-check can verify behaviour:
   `{"answer": str, "status": "HIT" | "MISS" | "DENIED", "role": str | None}`
4. Never cache a denial, and never cache a time-sensitive query. Reuse the
   `is_time_sensitive()` idea from `003. Agentic Router_semantic_caching_rbac.ipynb` — a stock
   price cached for an hour is a wrong answer served fast.
5. Make the leak test below pass.

**Hints**

- `SemanticCaching` in `rag_helpers.py` is a working FAISS cache — read it first.
  Partitioning it is mostly a matter of what you key and what you search.
- Think about what happens when a user's role *changes*. Should their old cache
  entries still be reachable? Write a sentence on your answer.
- Two roles share `OPENAI_QUERY`. A strictly per-role cache re-computes that answer
  once per role — correct, but wasteful. Caching per *permitted source* instead of per
  role fixes it. Trade-off worth a comment.

**Stretch:** log every request as an audit record — user, role, query, route, decision,
cache status, latency — and print a small table at the end. That table is what you'd
hand an auditor.


In [36]:
# FAISS for the bonus cache (the cache falls back to NumPy if this isn't available)
!pip install -q faiss-cpu

### Bonus — design choices

- **Partitioned, per knowledge source** (one FAISS index each for `OPENAI_QUERY`,
  `10K_DOCUMENT_QUERY`, `INTERNET_QUERY`). A lookup only *searches* partitions the caller's role
  may read, so a forbidden row is never even a candidate — safer than a tagged index where one
  post-filter is all that stands between a match and a leak.
- **Per source, not per role:** an answer comes from exactly one source, so the source decides who
  may read it. Both roles can read OpenAI docs, so that answer is computed once and reused by both
  (a per-role cache would compute it twice). Composite multi-source answers are never cached.
- **Role changes** take effect on the next request: permissions are read at lookup time and
  nothing is stored per user, so no purge is needed.
- **Order:** identity → route → permission → cache → pipeline. Denials and time-sensitive
  queries are never cached; writes are permission-scoped (no cache poisoning).


In [37]:
# ── Bonus: implementation ────────────────────────────────────────────────────
# Logic in src/cache.py (RoleAwareSemanticCache, is_time_sensitive) and src/pipeline.py
# (secure_agentic_rag_cached, audit log). Design rationale: see the markdown cell above
# and the RoleAwareSemanticCache docstring.
from src.cache import RoleAwareSemanticCache as _RoleAwareSemanticCache, is_time_sensitive, faiss
from src.pipeline import AUDIT_LOG, print_audit_log, secure_agentic_rag_cached as _secure_agentic_rag_cached

if faiss is None:
    print("faiss not installed — RoleAwareSemanticCache will use a NumPy search instead.")


class RoleAwareSemanticCache(_RoleAwareSemanticCache):
    """Partitioned-per-source semantic cache using the Nomic embedder from Section 4."""

    def __init__(self, threshold: float = 0.2, embed_fn=None):
        super().__init__(threshold=threshold, embed_fn=embed_fn or get_text_embeddings)


def secure_agentic_rag_cached(user_id: str, user_query: str, cache, verbose: bool = True) -> dict:
    """identity → route → permission → cache → pipeline. Returns {"answer", "status", "role", "route"}."""
    return _secure_agentic_rag_cached(ctx, user_id, user_query, cache, verbose=verbose)


In [38]:
# ── Bonus: self-check — this must pass ───────────────────────────────────────
# It asserts behaviour, not wording, so your answer text can be anything.

def run_self_check():
    cache = RoleAwareSemanticCache()
    q_fin = "what was uber revenue in 2021?"
    q_doc = "how do I build an agent with the OpenAI Agents SDK?"

    # 1. bob may read financials — first ask is a MISS
    r = secure_agentic_rag_cached("bob", q_fin, cache)
    assert r["status"] == "MISS", f"expected MISS, got {r['status']}"

    # 2. bob asks again — served from cache
    r = secure_agentic_rag_cached("bob", q_fin, cache)
    assert r["status"] == "HIT", f"expected HIT, got {r['status']}"

    # 3. THE LEAK TEST — alice must be denied, never served bob's cached answer
    r = secure_agentic_rag_cached("alice", q_fin, cache)
    assert r["status"] == "DENIED", f"LEAK: alice got {r['status']} on finance data"

    # 4. a near-paraphrase must also be denied, not semantically matched into bob's rows
    r = secure_agentic_rag_cached("alice", "how much revenue did Uber make in 2021?", cache)
    assert r["status"] == "DENIED", f"LEAK: alice got {r['status']} via paraphrase"

    # 5. unknown users are rejected outright
    r = secure_agentic_rag_cached("carol", q_doc, cache)
    assert r["status"] == "DENIED", f"expected DENIED for unknown user, got {r['status']}"

    # 6. a shared source still caches normally within a role
    assert secure_agentic_rag_cached("alice", q_doc, cache)["status"] == "MISS"
    assert secure_agentic_rag_cached("alice", q_doc, cache)["status"] == "HIT"

    print("✅ All checks passed — cache is fast and does not leak across roles.")


# run_self_check()

In [39]:
SELF_CHECK_PASSED = False
run_self_check()
SELF_CHECK_PASSED = True


👤 bob (finance_analyst) · what was uber revenue in 2021?
   MISS route=10K_DOCUMENT_QUERY  stored
   Uber’s revenue in 2021 was **$17.455 billion** (approximately **$17.5 billion**).[1]



👤 bob (finance_analyst) · what was uber revenue in 2021?
   HIT route=10K_DOCUMENT_QUERY  similarity=1.000
   Uber’s revenue in 2021 was **$17.455 billion** (approximately **$17.5 billion**).[1]



👤 alice (engineer) · what was uber revenue in 2021?
   DENIED route=10K_DOCUMENT_QUERY  role may not query 10-K financial filings



👤 alice (engineer) · how much revenue did Uber make in 2021?
   DENIED route=10K_DOCUMENT_QUERY  role may not query 10-K financial filings

👤 carol (UNKNOWN) · how do I build an agent with the OpenAI Agents SDK?
   DENIED route=-  unknown user



👤 alice (engineer) · how do I build an agent with the OpenAI Agents SDK?
   MISS route=OPENAI_QUERY  stored
   An agent in the OpenAI Agents SDK is built from three core components:

1. **Model** — the LLM that reasons and makes decisions  
2. **Tools** — functions or APIs the agent can call  
3. **Instruction…



👤 alice (engineer) · how do I build an agent with the OpenAI Agents SDK?
   HIT route=OPENAI_QUERY  similarity=1.000
   An agent in the OpenAI Agents SDK is built from three core components:

1. **Model** — the LLM that reasons and makes decisions  
2. **Tools** — functions or APIs the agent can call  
3. **Instruction…

✅ All checks passed — cache is fast and does not leak across roles.


In [40]:
# ── Bonus: extra checks beyond the required self-check ───────────────────────

def run_extra_checks():
    cache = RoleAwareSemanticCache()
    q_fin = "what was uber revenue in 2021?"

    # a) Time-sensitive questions are never cached — asking twice is two MISSes
    for _ in range(2):
        r = secure_agentic_rag_cached("alice", "what are the latest LLMs released this week?", cache)
        assert r["status"] in ("MISS", "DENIED"), f"time-sensitive query was served from cache: {r['status']}"

    # b) Denials are never cached
    before = len(cache)
    secure_agentic_rag_cached("alice", q_fin, cache)
    assert len(cache) == before, "a denial was written to the cache"

    # c) Role change takes effect immediately — no purge required
    secure_agentic_rag_cached("bob", q_fin, cache)            # bob caches a 10-K answer
    original_role = USERS["bob"]
    try:
        USERS["bob"] = "engineer"                             # bob moves to engineering
        hit, *_ = cache.check("bob", q_fin, source="10K_DOCUMENT_QUERY")
        assert not hit, "LEAK: bob still reads 10-K cache entries after losing the role"
    finally:
        USERS["bob"] = original_role
    hit, *_ = cache.check("bob", q_fin, source="10K_DOCUMENT_QUERY")
    assert hit, "bob should see the 10-K entry again once the role is restored"

    # d) Writes are scoped too — nobody can plant an answer in a partition they can't read
    try:
        cache.add("alice", "fake question", "fake answer", source="10K_DOCUMENT_QUERY")
        raise AssertionError("alice wrote into the 10-K partition")
    except PermissionError:
        pass

    print("✅ Extra checks passed — no caching of volatile answers or denials, "
          "role changes apply instantly, writes are permission-scoped.")

In [41]:
EXTRA_CHECKS_PASSED = False
run_extra_checks()
EXTRA_CHECKS_PASSED = True


Getting your response from the internet 🌐 ...
👤 alice (engineer) · what are the latest LLMs released this week?
   MISS route=INTERNET_QUERY  time-sensitive → not cached
   [1] LLMs Are Complicated Now
    Llama 3 with GLM 5.2 (released 1 week ago) DeepSeekv3.2 isn't fully implemented, same with KimiK2.6, GLM5.2+, DeepSeekv4 has no implementation, MiniMaxM3 not ...
    S…



Getting your response from the internet 🌐 ...
👤 alice (engineer) · what are the latest LLMs released this week?
   MISS route=INTERNET_QUERY  time-sensitive → not cached
   [1] LLMs Are Complicated Now
    Llama 3 with GLM 5.2 (released 1 week ago) DeepSeekv3.2 isn't fully implemented, same with KimiK2.6, GLM5.2+, DeepSeekv4 has no implementation, MiniMaxM3 not ...
    S…



👤 alice (engineer) · what was uber revenue in 2021?
   DENIED route=10K_DOCUMENT_QUERY  role may not query 10-K financial filings



👤 bob (finance_analyst) · what was uber revenue in 2021?
   MISS route=10K_DOCUMENT_QUERY  stored
   Uber’s revenue in 2021 was **$17.455 billion** (approximately **$17.5 billion**). [1]

✅ Extra checks passed — no caching of volatile answers or denials, role changes apply instantly, writes are permission-scoped.


In [42]:
# Audit trail for every request made through secure_agentic_rag_cached() above
print_audit_log()

user    | role             | route               | decision | cache  | latency_ms | query                                           
------------------------------------------------------------------------------------------------------------------------------------
bob     | finance_analyst  | 10K_DOCUMENT_QUERY  | ALLOW    | MISS   | 2621       | what was uber revenue in 2021?                  
bob     | finance_analyst  | 10K_DOCUMENT_QUERY  | ALLOW    | HIT    | 1115       | what was uber revenue in 2021?                  
alice   | engineer         | 10K_DOCUMENT_QUERY  | DENY     | DENIED | 866        | what was uber revenue in 2021?                  
alice   | engineer         | 10K_DOCUMENT_QUERY  | DENY     | DENIED | 1421       | how much revenue did Uber make in 2021?         
carol   | UNKNOWN          | -                   | DENY     | DENIED | 0          | how do I build an agent with the OpenAI Agents S
alice   | engineer         | OPENAI_QUERY        | ALLOW    | MISS   

---

# Evaluation

Everything in this section runs the **real implementation** against the live APIs — nothing is
hard-coded. LLM routing is not deterministic, so numbers can move slightly between runs. The two
offline suites (parser robustness, citations) use a scripted model so they can force bad output;
they are the same checks `python -m pytest` runs.


## 1. Routing evaluation

28 queries: OpenAI docs, 10-K, internet/current, ambiguous (more than one route is acceptable),
and mixed-domain compound questions. Each query is split and every sub-query routed (no retrieval).

- **Routing Accuracy** = correct route decisions / total route decisions
  (a query contributes max(expected, actual) decisions, so over-splitting is penalised)
- **Split Accuracy** = queries with the expected number of sub-queries / total queries
- **Latency** = split + routing time for the query


In [43]:
import pandas as pd
from IPython.display import display
from src import evaluation as ev

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 250)

routing_df, routing_summary = ev.run_routing_eval(ctx)

print(f"Routing Accuracy: {routing_summary['correct_route_decisions']}/{routing_summary['route_decisions']}"
      f" = {routing_summary['routing_accuracy']:.1%}")
print(f"Split Accuracy:   {int(routing_df['Split Correct'].sum())}/{len(routing_df)}"
      f" = {routing_summary['split_accuracy']:.1%}")
print(f"Mean latency (split + route): {routing_summary['mean_latency_s']:.2f}s\n")

display(routing_df[["Category", "Query", "Expected Subqueries", "Actual Subqueries", "Expected Routes",
                    "Actual Routes", "Split Correct", "Route Correct", "Latency"]])
display(routing_df.groupby("Category")[["Split Correct", "Route Correct"]].mean().style.format("{:.0%}"))


Routing Accuracy: 35/35 = 100.0%
Split Accuracy:   28/28 = 100.0%
Mean latency (split + route): 2.65s



,Category,Query,Expected Subqueries,Actual Subqueries,Expected Routes,Actual Routes,Split Correct,Route Correct,Latency
0,openai,What is an AI agent?,1,1,OPENAI_QUERY,OPENAI_QUERY,True,True,1.94
1,openai,How do I add guardrails to an agent with the OpenAI Agents SDK?,1,1,OPENAI_QUERY,OPENAI_QUERY,True,True,1.94
2,openai,How do handoffs between agents work in the OpenAI Agents SDK?,1,1,OPENAI_QUERY,OPENAI_QUERY,True,True,1.96
3,openai,How to work with chat completions?,1,1,OPENAI_QUERY,OPENAI_QUERY,True,True,3.03
4,openai,How do I use the OpenAI moderation API?,1,1,OPENAI_QUERY,OPENAI_QUERY,True,True,1.89
5,openai,How do I create embeddings with the OpenAI API?,1,1,OPENAI_QUERY,OPENAI_QUERY,True,True,2.06
6,10k,What was Uber revenue in 2021?,1,1,10K_DOCUMENT_QUERY,10K_DOCUMENT_QUERY,True,True,4.25
7,10k,What was Lyft's net loss in 2021?,1,1,10K_DOCUMENT_QUERY,10K_DOCUMENT_QUERY,True,True,2.50
8,10k,What risk factors did Uber list in its 2021 annual report?,1,1,10K_DOCUMENT_QUERY,10K_DOCUMENT_QUERY,True,True,1.57
9,10k,How many active riders did Lyft report in its 10-K?,1,1,10K_DOCUMENT_QUERY,10K_DOCUMENT_QUERY,True,True,3.63


,Split Correct,Route Correct
Category,,
10k,100%,100%
ambiguous,100%,100%
compound,100%,100%
internet,100%,100%
openai,100%,100%


In [44]:
# Any misses, with the actual sub-queries the splitter produced
misses = routing_df[~(routing_df["Split Correct"] & routing_df["Route Correct"])]
print(f"{len(misses)} of {len(routing_df)} queries missed")
display(misses[["Query", "Sub-queries", "Expected Routes", "Actual Routes"]])


0 of 28 queries missed


,Query,Sub-queries,Expected Routes,Actual Routes


## 2. Compound-query evaluation

**Live:** same-route, mixed-route, 3-part, duplicate, and a 7-question input that must be capped
at `MAX_SUB_QUERIES`. The first three run the full pipeline (retrieval + composition) and check
that every `[k.n]` in the final answer points at a real citation in sub-answer *k*.

**Offline (scripted model):** malformed, fenced, prose-wrapped, empty and `None` splitter/router
replies, a model without structured outputs, and an API error — none may crash.


In [45]:
compound_df = ev.run_compound_eval(ctx)
print(f"Live compound cases passed: {int(compound_df['Pass'].sum())}/{len(compound_df)}")
display(compound_df[["Case", "Expected Subqueries", "Actual Subqueries", "Expected Routes", "Actual Routes",
                     "Within Cap", "No Duplicates", "Citations Traceable", "Compose Method", "Pass"]])


Getting your response from the internet 🌐 ...


Getting your response from the internet 🌐 ...


Live compound cases passed: 5/5


,Case,Expected Subqueries,Actual Subqueries,Expected Routes,Actual Routes,Within Cap,No Duplicates,Citations Traceable,Compose Method,Pass
0,same-route,2,2,10K_DOCUMENT_QUERY | 10K_DOCUMENT_QUERY,10K_DOCUMENT_QUERY | 10K_DOCUMENT_QUERY,True,True,True,llm,True
1,mixed-route,2,2,10K_DOCUMENT_QUERY | INTERNET_QUERY,10K_DOCUMENT_QUERY | INTERNET_QUERY,True,True,True,llm,True
2,3-part,3,3,OPENAI_QUERY | 10K_DOCUMENT_QUERY | INTERNET_QUERY,OPENAI_QUERY | 10K_DOCUMENT_QUERY | INTERNET_QUERY,True,True,True,llm,True
3,duplicate,1,1,10K_DOCUMENT_QUERY,10K_DOCUMENT_QUERY,True,True,n/a,n/a (split + route only),True
4,over cap (7 questions → 5),5,5,OPENAI_QUERY | OPENAI_QUERY | OPENAI_QUERY | 10K_DOCUMENT_QUERY | 10K_DOCUMENT_QUERY,OPENAI_QUERY | OPENAI_QUERY | OPENAI_QUERY | 10K_DOCUMENT_QUERY | 10K_DOCUMENT_QUERY,True,True,n/a,n/a (split + route only),True


In [46]:
robustness_df = ev.run_parser_robustness_tests()
print(f"Parser robustness: {int(robustness_df['Pass'].sum())}/{len(robustness_df)} passed")
display(robustness_df[["Component", "Case", "Model Output", "Result", "Parsed By", "Pass"]])


⚠️ Split failed (503 Service Unavailable); treating the input as one query.
Parser robustness: 20/20 passed


,Component,Case,Model Output,Result,Parsed By,Pass
0,splitter,clean structured JSON,"'{""subQuestions"": [""What was Uber revenue in 2021?"", ""What a","[What was Uber revenue in 2021?, What are the newest LLMs?]",structured,True
1,splitter,Markdown code fence,"'```json\n{""subQuestions"": [""What was Uber revenue in 2021?""","[What was Uber revenue in 2021?, What are the newest LLMs?]",parser,True
2,splitter,prose around JSON,"'Sure! Here you go: {""subQuestions"": [""What was Uber revenue","[What was Uber revenue in 2021?, What are the newest LLMs?]",parser,True
3,splitter,bare JSON list,"'[""What was Uber revenue in 2021?"", ""What are the newest LLM","[What was Uber revenue in 2021?, What are the newest LLMs?]",parser,True
4,splitter,alternative key,"'{""sub_queries"": [""What was Uber revenue in 2021?"", ""What ar","[What was Uber revenue in 2021?, What are the newest LLMs?]",parser,True
5,splitter,duplicates + blanks,"'{""subQuestions"": [""What was Uber revenue in 2021?"", ""What w",[What was Uber revenue in 2021?],structured,True
6,splitter,over cap (8 → 5),"'{""subQuestions"": [""Question 0?"", ""Question 1?"", ""Question 2","[Question 0?, Question 1?, Question 2?, Question 3?, Question 4?]",structured,True
7,splitter,truncated JSON,"'{""subQuestions"": [""What was Uber revenue in 2021?"", '",[what was uber revenue in 2021 and what are the newest LLMs?],fallback,True
8,splitter,empty response,'',[what was uber revenue in 2021 and what are the newest LLMs?],fallback,True
9,splitter,None content,None,[what was uber revenue in 2021 and what are the newest LLMs?],fallback,True


## 3. Citation tests

Namespacing (`[1]` → `[k.1]`), no collisions across sub-answers, non-numeric labels such as
`[Direct Answer]` left alone, and the deterministic fallback when the composition model drops
citations or fails.


In [47]:
citation_df = ev.run_citation_tests()
print(f"Citation tests: {int(citation_df['Pass'].sum())}/{len(citation_df)} passed")
display(citation_df)


⚠️ Composition dropped citations for sub-question(s) [2]; using per-question sections.
⚠️ Composition failed (timeout); falling back to per-question sections.
Citation tests: 10/10 passed


,Test,Pass,Detail
0,[1] from sub-question 1 → [1.1],True,Revenue was $17.5B [1.1].
1,[1] from sub-question 2 → [2.1],True,Newest model is X [2.1].
2,identifiers do not collide,True,"[('1', '1'), ('1', '2')] vs [('2', '1')]"
3,[Direct Answer] label untouched,True,[Direct Answer] foo\n[2.1] Title
4,"grouped [1, 2] → [3.1][3.2]",True,Both agree [3.1][3.2].
5,"decimal tags untouched, multi-digit namespaced",True,See [1.5] and [1.12].
6,composition preserves namespaces,True,method=llm
7,dropped citations → deterministic composition,True,method=deterministic: dropped citations [2]
8,composition LLM error → deterministic composition,True,method=deterministic: LLM error
9,sub-answer without citations is not required to be cited,True,method=llm


## 4. Concurrency benchmark

Each compound query runs through `agentic_rag_multi()` once sequentially and once with
`concurrent=True` (order alternates per query). The split and the final composition are
sequential in both modes, so only the per-sub-query work can overlap.


In [48]:
bench_df, bench_summary = ev.run_concurrency_benchmark(ctx)
display(bench_df)
print(f"Average sequential latency: {bench_summary['avg_sequential_s']:.2f}s")
print(f"Average concurrent latency: {bench_summary['avg_concurrent_s']:.2f}s")
print(f"Average speedup:            ×{bench_summary['avg_speedup']:.2f}")
print(f"Concurrent faster in:       {bench_summary['concurrent_faster_in']} queries")
print(bench_summary["conclusion"])


Getting your response from the internet 🌐 ...


Getting your response from the internet 🌐 ...


Getting your response from the internet 🌐 ...


Getting your response from the internet 🌐 ...


Getting your response from the internet 🌐 ...


Getting your response from the internet 🌐 ...


,Query,Number of Subqueries,Sequential Time,Concurrent Time,Speedup
0,What was Lyft revenue in 2021 and what was Uber revenue in 2021?,2,8.34,7.37,1.13
1,What was Uber's 2021 revenue and what are the newest LLMs?,2,6.67,6.17,1.08
2,"How do I add guardrails with the OpenAI Agents SDK, what was Lyft's revenue in 2021, a...",3,14.71,9.68,1.52
3,What are OpenAI embeddings and who won the latest Super Bowl?,2,9.41,8.11,1.16


Average sequential latency: 9.78s
Average concurrent latency: 7.83s
Average speedup:            ×1.22
Concurrent faster in:       4/4 queries
Concurrent execution was faster on every measured query.


## 5. Cache / RBAC test matrix

A fresh cache, the real router and the real retrieval pipeline. `run_self_check()` and
`run_extra_checks()` above remain the required checks; this matrix lays the same guarantees
out as a table (plus cross-role reuse of a shared source).


In [49]:
cache_matrix_df = ev.run_cache_matrix(ctx, RoleAwareSemanticCache)
print(f"Cache / RBAC scenarios passed: {int(cache_matrix_df['Pass'].sum())}/{len(cache_matrix_df)}")
display(cache_matrix_df[["Scenario", "User", "Role", "Query Source", "Expected", "Actual", "Pass"]])


Getting your response from the internet 🌐 ...


Getting your response from the internet 🌐 ...


Cache / RBAC scenarios passed: 13/13


,Scenario,User,Role,Query Source,Expected,Actual,Pass
0,authorized MISS,bob,finance_analyst,10K_DOCUMENT_QUERY,MISS,MISS,True
1,authorized HIT,bob,finance_analyst,10K_DOCUMENT_QUERY,HIT,HIT,True
2,unauthorized DENIED,alice,engineer,10K_DOCUMENT_QUERY,DENIED,DENIED,True
3,paraphrased unauthorized DENIED,alice,engineer,10K_DOCUMENT_QUERY,DENIED,DENIED,True
4,unknown user DENIED,carol,UNKNOWN,-,DENIED,DENIED,True
5,shared source MISS,alice,engineer,OPENAI_QUERY,MISS,MISS,True
6,shared source reused across roles,bob,finance_analyst,OPENAI_QUERY,HIT,HIT,True
7,time-sensitive query bypass (2nd ask),alice,engineer,INTERNET_QUERY,"MISS, cache size 2","MISS, cache size 2",True
8,denied request not cached,alice,engineer,10K_DOCUMENT_QUERY,"DENIED, cache size 2","DENIED, cache size 2",True
9,role change (bob → engineer),bob,engineer,10K_DOCUMENT_QUERY,DENIED,DENIED,True


---

## Final Results

Built from the results computed above — re-running the notebook regenerates it.

In [50]:
def _status(ok) -> str:
    return "✅ Pass" if ok else "❌ Fail"

def _frac(mask) -> str:
    return f"{int(mask.sum())}/{len(mask)}"

single = routing_df[routing_df["Expected Subqueries"] == 1]
compound = routing_df[routing_df["Category"] == "compound"]
case = compound_df.set_index("Case")
multi_cases = compound_df[compound_df["Case"].isin(["same-route", "mixed-route", "3-part"])]
m = cache_matrix_df.set_index("Scenario")["Pass"]
isolation = ["unauthorized DENIED", "paraphrased unauthorized DENIED", "role change (bob → engineer)",
             "role change: direct cache read", "unauthorized cache write prevented", "denied request not cached"]

final_results = pd.DataFrame([
    ("Single-query routing", _status(single["Route Correct"].all()),
     f"routing eval: {_frac(single['Route Correct'])} single questions routed correctly"),
    ("Compound splitting", _status(compound["Split Correct"].all() and compound_df["Split Correct"].all()),
     f"routing eval {_frac(compound['Split Correct'])} compound splits; compound eval {_frac(compound_df['Split Correct'])}"),
    ("Same-source compound routing", _status(bool(case.loc["same-route", "Pass"])),
     f"same-route case: {case.loc['same-route', 'Actual Routes']}"),
    ("Mixed-source routing", _status(bool(case.loc["mixed-route", "Pass"] and case.loc["3-part", "Pass"])),
     f"mixed: {case.loc['mixed-route', 'Actual Routes']}; 3-part: {case.loc['3-part', 'Actual Routes']}"),
    ("Defensive parsing", _status(robustness_df["Pass"].all()),
     f"{_frac(robustness_df['Pass'])} malformed/fenced/prose/empty cases handled"),
    ("Citation preservation", _status(citation_df["Pass"].all() and (multi_cases["Citations Traceable"] == True).all()),
     f"citation tests {_frac(citation_df['Pass'])}; live traceable {_frac(multi_cases['Citations Traceable'] == True)}"),
    ("Concurrent execution", "✅ Works" + ("" if bench_summary["avg_speedup"] > 1 else " (no measured gain)"),
     f"avg ×{bench_summary['avg_speedup']:.2f}; faster in {bench_summary['concurrent_faster_in']} queries"),
    ("RBAC", _status(SELF_CHECK_PASSED and m[["unauthorized DENIED", "unknown user DENIED"]].all()),
     "run_self_check() passed; unauthorized + unknown users DENIED"),
    ("Semantic cache", _status(EXTRA_CHECKS_PASSED and m[["authorized MISS", "authorized HIT", "shared source reused across roles"]].all()),
     "MISS → HIT; shared OpenAI-docs answer reused across roles"),
    ("Cache isolation", _status(m[isolation].all()),
     f"{int(m[isolation].sum())}/{len(isolation)} isolation scenarios (paraphrase, role change, writes)"),
    ("Time-sensitive cache bypass", _status(bool(m["time-sensitive query bypass (2nd ask)"])),
     "latest-LLMs question asked twice → MISS both times, cache size unchanged"),
], columns=["Capability", "Status", "Evidence"])

print(f"Routing accuracy {routing_summary['routing_accuracy']:.1%} · split accuracy {routing_summary['split_accuracy']:.1%}")
display(final_results)


Routing accuracy 100.0% · split accuracy 100.0%


,Capability,Status,Evidence
0,Single-query routing,✅ Pass,routing eval: 22/22 single questions routed correctly
1,Compound splitting,✅ Pass,routing eval 6/6 compound splits; compound eval 5/5
2,Same-source compound routing,✅ Pass,same-route case: 10K_DOCUMENT_QUERY | 10K_DOCUMENT_QUERY
3,Mixed-source routing,✅ Pass,mixed: 10K_DOCUMENT_QUERY | INTERNET_QUERY; 3-part: OPENAI_QUERY | 10K_DOCUMENT_QUERY ...
4,Defensive parsing,✅ Pass,20/20 malformed/fenced/prose/empty cases handled
5,Citation preservation,✅ Pass,citation tests 10/10; live traceable 3/3
6,Concurrent execution,✅ Works,avg ×1.22; faster in 4/4 queries
7,RBAC,✅ Pass,run_self_check() passed; unauthorized + unknown users DENIED
8,Semantic cache,✅ Pass,MISS → HIT; shared OpenAI-docs answer reused across roles
9,Cache isolation,✅ Pass,"6/6 isolation scenarios (paraphrase, role change, writes)"


## Key Design Decisions

- **Split before routing.** The router picks exactly one source, so a compound question sent to it
  whole gets one route and a partial answer. Splitting first gives each question a route of its own.
- **Each sub-question routes independently.** Sub-questions often need different sources (10-K +
  internet). Routing them separately is what makes mixed-source answers possible; it also isolates
  failures, since one bad route doesn't spoil the others.
- **Citations are namespaced.** Every sub-answer numbers its sources from `[1]`, so merged answers
  would have colliding `[1]`s. Rewriting sub-answer *k*'s `[n]` to `[k.n]` keeps every claim traceable
  to its sub-question and source (see the citation key under each composed answer).
- **Composition has a deterministic fallback.** The composing LLM can drop or garble citations. The
  output is checked, and if any sub-answer's citations are missing (or the call fails) the answer is
  assembled as one section per sub-question, which is less fluent but never ungrounded.
- **Authorization happens before the cache.** A semantic cache matches on meaning, so a paraphrase
  could otherwise reach another role's entries. Checking permission first means a denied user never
  touches the cache, and a denial is never cached.
- **Volatile queries bypass the cache.** "Latest", "today", prices and similar go stale; serving them
  from cache would be a wrong answer returned quickly. The check is deliberately over-broad: a false
  positive costs one cache miss.
- **Structured output with fallbacks.** Router and splitter ask for schema-validated JSON first; the
  original defensive parser remains the second line, and a safe default the third, so a bad model
  reply degrades the answer instead of crashing the agent.
